# 📒 Notebook 5 — Hybrid Recommendation Engine

**This is the final inference notebook** — the engine that powers your dashboard.

## How the Hybrid Works:

### Existing User (has USER_ID in our system):
```
SVD score  (weight 0.6)
    +
KNN score  (weight 0.4)
    =
Hybrid score → sort → Top 3 by city/state
```
If a restaurant isn't in the training data → Pinecone fills the gap.

### New/Anonymous User (no history):
```
User describes preferences in text
    ↓
Pinecone semantic search (content-based)
    ↓
Filter by city/state → Top 3
```

**Requires:** Notebooks 2, 3, 4 completed (models saved in `models/` and `data/`)

## 1. Install Dependencies

In [2]:
!pip install scikit-surprise pinecone-client sentence-transformers pandas -q

## 2. Configuration

In [3]:
PINECONE_API_KEY    = "a349ad93-e142-410a-8cd7-5f7252092e12"
PINECONE_INDEX_NAME = "811-business-description"
EMBEDDING_MODEL     = "sentence-transformers/all-mpnet-base-v2"

# Hybrid weights (must sum to 1.0)
SVD_WEIGHT = 0.6
KNN_WEIGHT = 0.4

TOP_K = 3   # Number of recommendations to return

print("✅ Config loaded")

✅ Config loaded


## 3. Load All Models & Data

In [4]:
import pickle
import pandas as pd

# ── Load trained models ────────────────────────────────────────────────────────
with open("models/svd_model.pkl", "rb") as f:
    svd_model = pickle.load(f)
with open("models/knn_model.pkl", "rb") as f:
    knn_model = pickle.load(f)

# ── Load encoders & metadata ───────────────────────────────────────────────────
with open("data/user_encoder.pkl", "rb") as f:
    user_encoder = pickle.load(f)
with open("data/business_encoder.pkl", "rb") as f:
    biz_encoder = pickle.load(f)

train_df      = pd.read_csv("data/ratings_train.csv")
business_meta = pd.read_csv("data/business_meta.csv")

print("✅ SVD model loaded")
print("✅ KNN model loaded")
print(f"✅ Business metadata: {len(business_meta):,} restaurants")
print(f"✅ Training data   : {len(train_df):,} ratings")

✅ SVD model loaded
✅ KNN model loaded
✅ Business metadata: 33,016 restaurants
✅ Training data   : 1,579,941 ratings


## 4. Connect to Pinecone (for new users & fallback)

In [5]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

pc       = Pinecone(api_key=PINECONE_API_KEY)
pine_idx = pc.Index(PINECONE_INDEX_NAME)

embedder = SentenceTransformer(EMBEDDING_MODEL)

stats = pine_idx.describe_index_stats()
print(f"✅ Pinecone connected | Vectors in index: {stats.get('total_vector_count', 0):,}")

/opt/anaconda3/envs/PAQIDB/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Pinecone connected | Vectors in index: 99


## 5. Helper: Pinecone Search (Content-Based)
Used for new users AND as a fallback for restaurants not in training data.

In [6]:
def pinecone_search(query_text, city=None, state=None, top_k=10):
    """
    Semantic search in Pinecone.
    Returns a DataFrame with matching restaurants + similarity scores.
    """
    vec = embedder.encode([query_text], normalize_embeddings=True)[0].tolist()

    # Build metadata filter
    pine_filter = {}
    if city and state:
        pine_filter = {
            "$and": [
                {"city":  {"$eq": city}},
                {"state": {"$eq": state.upper()}}
            ]
        }
    elif city:
        pine_filter = {"city": {"$eq": city}}
    elif state:
        pine_filter = {"state": {"$eq": state.upper()}}

    results = pine_idx.query(
        vector=vec,
        top_k=top_k,
        include_metadata=True,
        filter=pine_filter if pine_filter else None
    )

    rows = []
    for m in results["matches"]:
        md = m["metadata"]
        rows.append({
            "business_id":      m["id"],
            "business_name":    md.get("business_name", "N/A"),
            "city":             md.get("city", "N/A"),
            "state":            md.get("state", "N/A"),
            "avg_stars":        md.get("avg_stars", 0),
            "primary_category": md.get("primary_category", "N/A"),
            "description":      md.get("description", "N/A"),
            "pinecone_score":   m["score"]
        })

    return pd.DataFrame(rows)

print("✅ Pinecone search helper ready")

✅ Pinecone search helper ready


## 6. Core: Hybrid Recommendation for EXISTING Users

In [7]:
def normalize_scores(series):
    """Min-max normalize a score series to [0, 1]."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return series * 0 + 1.0   # All same score → return 1
    return (series - mn) / (mx - mn)


def hybrid_recommend_existing(user_id, city=None, state=None, top_k=3, preference_text=None):
    """
    Hybrid recommendation for an existing user.
    
    Args:
        user_id        : String user ID from the database
        city           : Filter by city (e.g. 'Philadelphia')
        state          : Filter by state (e.g. 'PA')
        top_k          : Number of results
        preference_text: Optional — if provided, also boosts results matching this text
                         via Pinecone (e.g. 'spicy Mexican food')
    """
    print(f"\n🔍 Hybrid recommendation for existing user: {user_id[:12]}...")
    print(f"   Location filter: {city or 'any'}, {state or 'any'}")

    # ── Get rated restaurants (to exclude from recommendations) ────────────────
    rated_biz = set(train_df[train_df["user_id"] == user_id]["business_id"].tolist())

    # ── Filter candidate businesses by location ─────────────────────────────────
    candidates = business_meta.copy()
    if city:
        candidates = candidates[candidates["city"].str.lower() == city.lower()]
    if state:
        candidates = candidates[candidates["state"].str.upper() == state.upper()]
    candidates = candidates[~candidates["business_id"].isin(rated_biz)].reset_index(drop=True)

    if candidates.empty:
        print(f"⚠️  No candidates in {city}, {state}. Falling back to Pinecone only.")
        if preference_text:
            return _pinecone_fallback(preference_text, city, state, top_k)
        return pd.DataFrame()

    print(f"   Candidates to score: {len(candidates):,} restaurants")

    # ── Get SVD + KNN scores for each candidate ─────────────────────────────────
    svd_scores, knn_scores = [], []
    for _, biz in candidates.iterrows():
        svd_scores.append(svd_model.predict(user_id, biz["business_id"]).est)
        knn_scores.append(knn_model.predict(user_id, biz["business_id"]).est)

    candidates["svd_score"] = svd_scores
    candidates["knn_score"] = knn_scores

    # ── Normalize scores to [0,1] then compute weighted hybrid ─────────────────
    candidates["svd_norm"]    = normalize_scores(candidates["svd_score"])
    candidates["knn_norm"]    = normalize_scores(candidates["knn_score"])
    candidates["hybrid_score"] = (
        SVD_WEIGHT * candidates["svd_norm"] +
        KNN_WEIGHT * candidates["knn_norm"]
    )

    # ── Optional: boost with Pinecone semantic score if preference_text given ───
    if preference_text:
        pine_results = pinecone_search(preference_text, city=city, state=state, top_k=50)
        if not pine_results.empty:
            pine_dict = dict(zip(pine_results["business_id"], pine_results["pinecone_score"]))
            candidates["pine_score"] = candidates["business_id"].map(pine_dict).fillna(0)
            candidates["pine_norm"]  = normalize_scores(candidates["pine_score"])
            # Blend: SVD 0.5 + KNN 0.3 + Pinecone 0.2 when preference_text given
            candidates["hybrid_score"] = (
                0.5 * candidates["svd_norm"] +
                0.3 * candidates["knn_norm"] +
                0.2 * candidates["pine_norm"]
            )
            print("   Boosted with Pinecone semantic score (preference text provided)")

    # ── Sort and return top K ───────────────────────────────────────────────────
    top = candidates.sort_values("hybrid_score", ascending=False).head(top_k).reset_index(drop=True)

    print(f"\n🎯 Top-{top_k} Hybrid Recommendations")
    print("=" * 65)
    for i, row in top.iterrows():
        print(f"\n#{i+1} {row['business_name']}")
        print(f"   📍 {row['city']}, {row['state']}")
        print(f"   🍽️  {row['categories']}")
        print(f"   ⭐ Avg Stars  : {row['business_avg_stars']}")
        print(f"   📊 SVD Score  : {row['svd_score']:.3f}")
        print(f"   📊 KNN Score  : {row['knn_score']:.3f}")
        print(f"   🏆 Hybrid Score: {row['hybrid_score']:.4f}")

    return top


print("✅ Hybrid recommendation function ready")

✅ Hybrid recommendation function ready


## 7. Core: Pinecone-Only Recommendation for NEW Users

In [8]:
def _pinecone_fallback(preference_text, city, state, top_k):
    """Internal fallback used when candidates list is empty."""
    df = pinecone_search(preference_text, city=city, state=state, top_k=top_k)
    return df.head(top_k)


def recommend_new_user(preference_text, city=None, state=None, top_k=3):
    """
    Recommendation for a NEW user with no history.
    Uses Pinecone semantic search only.
    
    Args:
        preference_text: What the user likes (e.g. 'cozy Italian restaurant with great pasta')
        city           : Filter by city
        state          : Filter by state
        top_k          : Number of results
    """
    print(f"\n🆕 New user recommendation")
    print(f"   Preference : '{preference_text}'")
    print(f"   Location   : {city or 'any'}, {state or 'any'}")

    results = pinecone_search(preference_text, city=city, state=state, top_k=top_k)

    if results.empty:
        print(f"⚠️  No results found. Try a different city/state or broaden your preferences.")
        return results

    print(f"\n🎯 Top-{top_k} Content-Based Recommendations")
    print("=" * 65)
    for i, row in results.head(top_k).iterrows():
        print(f"\n#{i+1} {row['business_name']}")
        print(f"   📍 {row['city']}, {row['state']}")
        print(f"   🍽️  {row['primary_category']}")
        print(f"   ⭐ Avg Stars      : {row['avg_stars']}")
        print(f"   🔍 Similarity Score: {row['pinecone_score']:.4f}")
        print(f"   📝 {row['description']}")

    return results.head(top_k)


print("✅ New user recommendation function ready")

✅ New user recommendation function ready


## 8. Master Recommend Function
This is the **single entry point** your dashboard will call.

In [9]:
def recommend(
    user_id=None,
    preference_text=None,
    city=None,
    state=None,
    top_k=3
):
    """
    Master recommendation function — auto-routes to correct strategy.

    Args:
        user_id        : Existing user ID (or None for new user)
        preference_text: Text description of preferences (used for new users
                         and as optional boost for existing users)
        city           : City to search in
        state          : State to search in
        top_k          : Number of recommendations

    Returns:
        DataFrame of top-K recommendations
    """
    if user_id and user_id in user_encoder["str2idx"]:
        # ── EXISTING USER: Use hybrid SVD + KNN (+ optional Pinecone boost) ────
        return hybrid_recommend_existing(
            user_id=user_id,
            city=city,
            state=state,
            top_k=top_k,
            preference_text=preference_text
        )
    else:
        if user_id:
            print(f"ℹ️  User '{user_id}' not found in training data — treating as new user.")
        if not preference_text:
            print("⚠️  New user with no preference text — please provide preference_text.")
            return pd.DataFrame()
        # ── NEW USER: Use Pinecone semantic search ───────────────────────────────
        return recommend_new_user(
            preference_text=preference_text,
            city=city,
            state=state,
            top_k=top_k
        )


print("✅ Master recommend() function ready")

✅ Master recommend() function ready


## 9. Test All Scenarios

In [10]:
# ── Scenario 1: Existing user, location filter ─────────────────────────────────
print("\n" + "#" * 70)
print("SCENARIO 1: Existing user with location filter")
print("#" * 70)

sample_user  = train_df["user_id"].iloc[0]
sample_city  = business_meta["city"].value_counts().index[0]
sample_state = business_meta[business_meta["city"] == sample_city]["state"].iloc[0]

recommend(
    user_id=sample_user,
    city=sample_city,
    state=sample_state,
    top_k=3
)


######################################################################
SCENARIO 1: Existing user with location filter
######################################################################

🔍 Hybrid recommendation for existing user: 6jz_Yr6_AP2W...
   Location filter: Philadelphia, PA
   Candidates to score: 3,918 restaurants

🎯 Top-3 Hybrid Recommendations

#1 Gran Caffe L'Aquila
   📍 Philadelphia, PA
   🍽️  Restaurants, Gelato, Coffee & Tea, Food, Italian, Bakeries
   ⭐ Avg Stars  : 4.5
   📊 SVD Score  : 5.000
   📊 KNN Score  : 4.000
   🏆 Hybrid Score: 1.0000

#2 Khmer Kitchen
   📍 Philadelphia, PA
   🍽️  Restaurants, Cambodian
   ⭐ Avg Stars  : 4.5
   📊 SVD Score  : 4.958
   📊 KNN Score  : 4.000
   🏆 Hybrid Score: 0.9923

#3 Morimoto
   📍 Philadelphia, PA
   🍽️  Japanese, American (Traditional), American (New), Restaurants, Sushi Bars, Asian Fusion
   ⭐ Avg Stars  : 4.5
   📊 SVD Score  : 4.925
   📊 KNN Score  : 4.000
   🏆 Hybrid Score: 0.9864


,business_id,business_name,city,state,business_avg_stars,categories,svd_score,knn_score,svd_norm,knn_norm,hybrid_score
0,-cEFKAznWmI0cledNOIQ7w,Gran Caffe L'Aquila,Philadelphia,PA,4.5,"Restaurants, Gelato, Coffee & Tea, Food, Itali...",5.000000,4.0,1.000000,1.0,1.000000
1,7yPzOek7Af2dKWGYKpcHWA,Khmer Kitchen,Philadelphia,PA,4.5,"Restaurants, Cambodian",4.957733,4.0,0.987215,1.0,0.992329
2,6_T2xzR74JqGCTPefAD8Tw,Morimoto,Philadelphia,PA,4.5,"Japanese, American (Traditional), American (Ne...",4.925138,4.0,0.977355,1.0,0.986413


In [11]:
# ── Scenario 2: Existing user with preference text + location ──────────────────
print("\n" + "#" * 70)
print("SCENARIO 2: Existing user + preference text (hybrid with Pinecone boost)")
print("#" * 70)

recommend(
    user_id=sample_user,
    preference_text="cozy Italian restaurant with great pasta and wine",
    city=sample_city,
    state=sample_state,
    top_k=3
)


######################################################################
SCENARIO 2: Existing user + preference text (hybrid with Pinecone boost)
######################################################################

🔍 Hybrid recommendation for existing user: 6jz_Yr6_AP2W...
   Location filter: Philadelphia, PA
   Candidates to score: 3,918 restaurants
   Boosted with Pinecone semantic score (preference text provided)

🎯 Top-3 Hybrid Recommendations

#1 Blue Cat Restaurant
   📍 Philadelphia, PA
   🍽️  Latin American, American (New), Restaurants, Sandwiches
   ⭐ Avg Stars  : 4.0
   📊 SVD Score  : 4.047
   📊 KNN Score  : 4.000
   🏆 Hybrid Score: 0.8316

#2 Eatalia
   📍 Philadelphia, PA
   🍽️  Italian, Restaurants
   ⭐ Avg Stars  : 4.0
   📊 SVD Score  : 3.876
   📊 KNN Score  : 4.000
   🏆 Hybrid Score: 0.8300

#3 Gran Caffe L'Aquila
   📍 Philadelphia, PA
   🍽️  Restaurants, Gelato, Coffee & Tea, Food, Italian, Bakeries
   ⭐ Avg Stars  : 4.5
   📊 SVD Score  : 5.000
   📊 KNN Score  : 4.000
 

,business_id,business_name,city,state,business_avg_stars,categories,svd_score,knn_score,svd_norm,knn_norm,hybrid_score,pine_score,pine_norm
0,rvbf15JuvNJyhO2Q5zIapQ,Blue Cat Restaurant,Philadelphia,PA,4.0,"Latin American, American (New), Restaurants, S...",4.046841,4.0,0.711681,1.0,0.831572,0.549938,0.878654
1,DXYhpEv17mWqSqFjau7vhg,Eatalia,Philadelphia,PA,4.0,"Italian, Restaurants",3.875868,4.0,0.659964,1.0,0.829982,0.625887,1.000000
2,-cEFKAznWmI0cledNOIQ7w,Gran Caffe L'Aquila,Philadelphia,PA,4.5,"Restaurants, Gelato, Coffee & Tea, Food, Itali...",5.000000,4.0,1.000000,1.0,0.800000,0.000000,0.000000


In [12]:
# ── Scenario 3: NEW user (no user_id) ─────────────────────────────────────────
print("\n" + "#" * 70)
print("SCENARIO 3: New user — Pinecone only")
print("#" * 70)

recommend(
    user_id=None,
    preference_text="spicy Mexican food with good margaritas and lively atmosphere",
    city=sample_city,
    state=sample_state,
    top_k=3
)


######################################################################
SCENARIO 3: New user — Pinecone only
######################################################################

🆕 New user recommendation
   Preference : 'spicy Mexican food with good margaritas and lively atmosphere'
   Location   : Philadelphia, PA

🎯 Top-3 Content-Based Recommendations

#1 Growlers
   📍 Philadelphia, PA
   🍽️  Gastropubs
   ⭐ Avg Stars      : 4.0
   🔍 Similarity Score: 0.5678
   📝 Awesome gastropub with delicious small plates, great beer list, and friendly service.

#2 Blue Cat Restaurant
   📍 Philadelphia, PA
   🍽️  Restaurants
   ⭐ Avg Stars      : 4.0
   🔍 Similarity Score: 0.4470
   📝 Blue Cat Restaurant in Philadelphia, PA offers good food and a cool vibe. Byob and parking available.

#3 El Provocon
   📍 Philadelphia, PA
   🍽️  Budget Restaurants
   ⭐ Avg Stars      : 2.5
   🔍 Similarity Score: 0.4434
   📝 El Provocon is a small, nice place where you can enjoy a 5 dollar lunch special, includi

,business_id,business_name,city,state,avg_stars,primary_category,description,pinecone_score
0,j4DHb4tmK3QlK0gqpdnMeA,Growlers,Philadelphia,PA,4.0,Gastropubs,"Awesome gastropub with delicious small plates,...",0.567812
1,rvbf15JuvNJyhO2Q5zIapQ,Blue Cat Restaurant,Philadelphia,PA,4.0,Restaurants,"Blue Cat Restaurant in Philadelphia, PA offers...",0.446957
2,qONVcsU_vo3KFve4PtZqpg,El Provocon,Philadelphia,PA,2.5,Budget Restaurants,"El Provocon is a small, nice place where you c...",0.443407


In [13]:
# ── Scenario 4: Unknown user ID (treated as new user) ─────────────────────────
print("\n" + "#" * 70)
print("SCENARIO 4: Unknown user ID → auto-routed to new user path")
print("#" * 70)

recommend(
    user_id="some-unknown-user-id-123",
    preference_text="sushi bar with fresh fish and minimalist decor",
    state=sample_state,
    top_k=3
)


######################################################################
SCENARIO 4: Unknown user ID → auto-routed to new user path
######################################################################
ℹ️  User 'some-unknown-user-id-123' not found in training data — treating as new user.

🆕 New user recommendation
   Preference : 'sushi bar with fresh fish and minimalist decor'
   Location   : any, PA

🎯 Top-3 Content-Based Recommendations

#1 From The Boot
   📍 Blue Bell, PA
   🍽️  Italian, Pizza, Fast Food
   ⭐ Avg Stars      : 4.0
   🔍 Similarity Score: 0.5422
   📝 Small restaurant with terrific food but no reservations. Atmosphere is fast food-like.

#2 Growlers
   📍 Philadelphia, PA
   🍽️  Gastropubs
   ⭐ Avg Stars      : 4.0
   🔍 Similarity Score: 0.5244
   📝 Awesome gastropub with delicious small plates, great beer list, and friendly service.

#3 Pure Fare
   📍 Philadelphia, PA
   🍽️  Vegan
   ⭐ Avg Stars      : 2.5
   🔍 Similarity Score: 0.5105
   📝 Small vegan restaurant with 

,business_id,business_name,city,state,avg_stars,primary_category,description,pinecone_score
0,EEQJ6wnHD0x1OM41ldTTWg,From The Boot,Blue Bell,PA,4.0,"Italian, Pizza, Fast Food",Small restaurant with terrific food but no res...,0.542236
1,j4DHb4tmK3QlK0gqpdnMeA,Growlers,Philadelphia,PA,4.0,Gastropubs,"Awesome gastropub with delicious small plates,...",0.524402
2,8ZhC29t_mepcAQ4OlN15NQ,Pure Fare,Philadelphia,PA,2.5,Vegan,Small vegan restaurant with limited sandwich a...,0.510536


---
## ✅ Hybrid Engine Complete!

| Scenario | Strategy | Models Used |
|----------|----------|-------------|
| Existing user, no preference | Hybrid | SVD (0.6) + KNN (0.4) |
| Existing user + preference text | Hybrid + Boost | SVD (0.5) + KNN (0.3) + Pinecone (0.2) |
| New user | Content-based | Pinecone only |
| Unknown user ID | Auto-routes → New user | Pinecone only |

**Next Step:** Build the Streamlit dashboard that calls `recommend()` with inputs from the UI.